<a href="https://colab.research.google.com/github/Moksh45/Run-and-Share-ComfyUI-on-Google-Colab/blob/main/run-and-share-comfyui-on-google-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Running ComfyUI in Google Colab

To run ComfyUI in Google Colab, you need to enable GPU acceleration. Go to "Runtime" -> "Change runtime type" and select "GPU" as the hardware accelerator.

In [ ]:
# Update package lists and install necessary packages
!apt-get update
!apt-get install -y wget aria2 libgl1-mesa-glx

In [ ]:
# Clone the ComfyUI repository from GitHub
!git clone https://github.com/comfyanonymous/ComfyUI.git

In [ ]:
# Change the current directory to ComfyUI and install required Python packages
%cd ComfyUI
!pip install -r requirements.txt

In [ ]:
# Install the pinggy library for creating public URLs
!pip install pinggy

In [ ]:
# Download LTX-2.3 models, LoRAs, and Text Encoders using huggingface_hub to prevent 403 Forbidden errors
import os
import shutil
import requests
from huggingface_hub import hf_hub_download

print("=> Criando a estrutura de pastas correta do ComfyUI...")
os.makedirs("./models/checkpoints", exist_ok=True)
os.makedirs("./models/latent_upscale_models", exist_ok=True)
os.makedirs("./models/loras", exist_ok=True)
os.makedirs("./models/text_encoders", exist_ok=True)
os.makedirs("/content/ComfyUI/input", exist_ok=True)

print("=> Baixando Checkpoints e Upscalers via HuggingFace Hub (Sem erro 403)...")
hf_hub_download(
    repo_id="Lightricks/LTX-2.3-fp8",
    filename="ltx-2.3-22b-dev-fp8.safetensors",
    local_dir="./models/checkpoints"
)

hf_hub_download(
    repo_id="Lightricks/LTX-2.3",
    filename="ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
    local_dir="./models/latent_upscale_models"
)

print("=> Baixando LoRAs...")
hf_hub_download(
    repo_id="Comfy-Org/ltx-2.3",
    filename="split_files/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors",
    local_dir="./models/loras",
    local_dir_use_symlinks=False
)
if os.path.exists("./models/loras/split_files/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors"):
    shutil.move("./models/loras/split_files/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors", "./models/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors")

hf_hub_download(
    repo_id="Comfy-Org/ltx-2",
    filename="split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors",
    local_dir="./models/loras",
    local_dir_use_symlinks=False
)
if os.path.exists("./models/loras/split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors"):
    shutil.move("./models/loras/split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors", "./models/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors")

print("=> Baixando Text Encoders...")
hf_hub_download(
    repo_id="Comfy-Org/ltx-2",
    filename="split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors",
    local_dir="./models/text_encoders",
    local_dir_use_symlinks=False
)
if os.path.exists("./models/text_encoders/split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors"):
    shutil.move("./models/text_encoders/split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors", "./models/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors")

if os.path.exists("./models/loras/split_files"):
    shutil.rmtree("./models/loras/split_files")
if os.path.exists("./models/text_encoders/split_files"):
    shutil.rmtree("./models/text_encoders/split_files")

print("=> Criando imagem fake de entrada para o nó LoadImage 269...")
url_placeholder = "https://raw.githubusercontent.com/comfyanonymous/ComfyUI/master/input/example.png"
destino_imagem = "/content/ComfyUI/input/egyptian_queen.png"
try:
    res = requests.get(url_placeholder)
    if res.status_code == 200:
        with open(destino_imagem, 'wb') as f:
            f.write(res.content)
        print("=> Imagem 'egyptian_queen.png' criada com sucesso!")
except Exception as e:
    print(f"Aviso: {e}")

print("=> All dependencies downloaded successfully!")

In [ ]:
# Start a pinggy tunnel to forward traffic from a public URL to localhost:8188
import pinggy
tunnel1 = pinggy.start_tunnel(
    forwardto="localhost:8188",
)

# Print the URLs of the started tunnel
print(f"Tunnel1 started - URLs: {tunnel1.urls}")

In [ ]:
# Executa o ComfyUI com otimizações extremas de VRAM para evitar quedas (Crash) no Colab T4
!python main.py --listen 0.0.0.0 --lowvram --gpu-only --fast --disable-smart-memory